# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [1]:
# Setup — loads .env, sets up Gemini client, defines Ollama fallback
import os, json, re, time, urllib.request
from google import genai
from google.genai import types

# ── Load .env (supports both KEY=val and export KEY=val) ──────────────────
def load_dotenv(path=".env"):
    if not os.path.exists(path):
        return
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[len("export "):]
            if "=" in line:
                key, _, val = line.partition("=")
                os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))

load_dotenv(".env")

# ── Gemini client (may be None if key missing) ────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GEMINI_MODEL   = "gemini-2.0-flash-lite"   # lite tier has higher free quota
OLLAMA_MODEL   = "llama3.2:3b"
OLLAMA_URL     = "http://localhost:11434/v1/chat/completions"

gemini_client = None
if GEMINI_API_KEY:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# ── Unified call: tries Gemini first, falls back to Ollama on 429 / no key ─
def llm_call(system: str, user: str, temperature: float = 0.7) -> tuple[str, str]:
    """
    Returns (response_text, backend_used).
    backend_used is 'gemini' or 'ollama'.
    """
    # ── Try Gemini ────────────────────────────────────────────────────────
    if gemini_client:
        try:
            r = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=user,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    temperature=temperature,
                ),
            )
            return r.text.strip(), "gemini"
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print(f"  ⚠ Gemini quota hit — falling back to Ollama")
            else:
                print(f"  ⚠ Gemini error ({e.__class__.__name__}) — falling back to Ollama")

    # ── Fall back to Ollama ───────────────────────────────────────────────
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "temperature": temperature,
        "stream": False,
    }).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        text = json.loads(resp.read())["choices"][0]["message"]["content"]
    return text.strip(), "ollama"


# ── Gemini call that also returns token usage (Ollama returns None) ────────
def llm_call_with_usage(system: str, user: str, temperature: float = 0.7):
    """
    Returns (response_text, usage_dict_or_None, backend_used).
    usage_dict keys: prompt_tokens, candidate_tokens, total_tokens.
    """
    if gemini_client:
        try:
            r = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=user,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    temperature=temperature,
                ),
            )
            u = r.usage_metadata
            usage = {
                "prompt_tokens":    u.prompt_token_count,
                "candidate_tokens": u.candidates_token_count,
                "total_tokens":     u.total_token_count,
            }
            return r.text.strip(), usage, "gemini"
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("  ⚠ Gemini quota hit — falling back to Ollama (no token usage available)")
            else:
                print(f"  ⚠ Gemini error — falling back to Ollama")

    text, backend = llm_call(system, user, temperature)
    # Ollama OpenAI-compat endpoint doesn't surface token counts reliably
    return text, None, backend


active = "gemini" if gemini_client else "ollama"
print(f"Setup complete. Primary backend: {active.upper()}")
print(f"  Gemini model : {GEMINI_MODEL}")
print(f"  Ollama model : {OLLAMA_MODEL} (fallback)")

Setup complete. Primary backend: GEMINI
  Gemini model : gemini-2.0-flash-lite
  Ollama model : llama3.2:3b (fallback)


## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [2]:
# ── Task 1a: single call + token usage ───────────────────────────────────

SYSTEM_1 = "You are a concise support assistant."
USER_1   = "A customer wrote: 'I was charged twice this month.' In one sentence, what should support do?"

text, usage, backend = llm_call_with_usage(SYSTEM_1, USER_1, temperature=0.7)

print(f"=== Response (via {backend.upper()}) ===")
print(text)
print()
print("=== Token Usage ===")
if usage:
    print(f"  Prompt tokens   : {usage['prompt_tokens']}")
    print(f"  Candidate tokens: {usage['candidate_tokens']}")
    print(f"  Total tokens    : {usage['total_tokens']}")
else:
    print("  (token counts not available from Ollama endpoint)")

  ⚠ Gemini quota hit — falling back to Ollama (no token usage available)
  ⚠ Gemini quota hit — falling back to Ollama
=== Response (via OLLAMA) ===
Inform the customer that they will need to contact their bank or payment provider to investigate and resolve the duplicate charge issue as our system is not responsible for processing transactions.

=== Token Usage ===
  (token counts not available from Ollama endpoint)


In [3]:
# ── Task 1b: same prompt × 3 at temp 0.1, then × 3 at temp 1.0 ──────────

PROMPT = "Name one creative way a software company can reduce customer churn."

for temp in [0.1, 1.0]:
    print(f"\n{'='*60}")
    print(f"  TEMPERATURE = {temp}")
    print(f"{'='*60}")
    for run in range(1, 4):
        text, backend = llm_call(SYSTEM_1, PROMPT, temperature=temp)
        print(f"\nRun {run} [{backend}]: {text}")


  TEMPERATURE = 0.1
  ⚠ Gemini quota hit — falling back to Ollama

Run 1 [ollama]: One creative way a software company can reduce customer churn is by implementing a "Success Plan" or "Onboarding Accelerator". This involves assigning a dedicated account manager to work closely with the customer, creating a personalized plan to ensure they're getting the most out of the software.

The plan could include:

* Regular check-ins and progress updates
* Customized training sessions or workshops
* Access to exclusive resources, such as expert webinars or online tutorials
* Quarterly review meetings to assess success and identify areas for improvement

By providing a tailored support experience, companies can increase customer satisfaction, reduce frustration, and ultimately decrease churn rates.
  ⚠ Gemini quota hit — falling back to Ollama

Run 2 [ollama]: One creative way a software company can reduce customer churn is by implementing a "Success Plan" or "Onboarding Accelerator". This invol

**What changed, and why?**

At **temperature 0.1** the model is nearly deterministic: probabilities are sharply
peaked so the highest-probability token almost always wins, giving near-identical
outputs across all three runs.
At **temperature 1.0** the raw logits are used unchanged, giving lower-probability
tokens a real chance — outputs diverge noticeably in wording, structure, and idea.
This maps directly to the sampling lesson: temperature *rescales* the logit
distribution before softmax. Low temp → peaked distribution → low entropy →
consistent. High temp → flat distribution → high entropy → creative/varied.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [4]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

ZERO_SHOT_SYSTEM = """You are a support-ticket classifier.
Classify the customer message into exactly one of these labels:
billing, bug, feature_request, other.
Reply with the label only — no punctuation, no explanation."""

FEW_SHOT_SYSTEM = """You are a support-ticket classifier.
Classify the customer message into exactly one of these labels:
billing, bug, feature_request, other.
Reply with the label only — no punctuation, no explanation.

Examples:
Message: "I was charged twice last month."
Label: billing

Message: "The app crashes when I open the settings page."
Label: bug

Message: "Please add a bulk-export feature to the dashboard."
Label: feature_request

Message: "Just wanted to say your team is fantastic!"
Label: other"""

COT_SYSTEM = """You are a support-ticket classifier.
Available labels: billing, bug, feature_request, other.

For each message:
1. Identify the core problem in one sentence.
2. Ask: Is money/payments involved? → billing
         Is something broken/not working? → bug
         Is a new capability requested? → feature_request
         Otherwise → other
3. Output your final answer on the last line as: LABEL: <label>"""

SYSTEM_MAP = {
    "zero_shot": ZERO_SHOT_SYSTEM,
    "few_shot":  FEW_SHOT_SYSTEM,
    "cot":       COT_SYSTEM,
}


def classify(text: str, style: str) -> str:
    system = SYSTEM_MAP[style]
    user   = f'Message: "{text}"'
    raw, _ = llm_call(system, user, temperature=0.2)
    raw    = raw.lower()

    # CoT ends with "LABEL: <label>" — extract just the label
    if style == "cot" and "label:" in raw:
        raw = raw.split("label:")[-1].strip()

    for label in LABELS:
        if label in raw:
            return label
    return "other"


styles  = ["zero_shot", "few_shot", "cot"]
results = []

for ticket in tickets:
    row = {"id": ticket["id"], "text": ticket["text"][:55] + "…", "truth": ticket["label"]}
    for style in styles:
        row[style] = classify(ticket["text"], style)
    results.append(row)

# ── Comparison table ───────────────────────────────────────────────────────
header = f"{'ID':<4} {'Ticket (truncated)':<57} {'Truth':<16} {'Zero-shot':<16} {'Few-shot':<16} {'CoT':<16}"
print(header)
print("-" * len(header))
for r in results:
    def m(s): return r[s] + (" ✓" if r[s] == r["truth"] else " ✗")
    print(f"{r['id']:<4} {r['text']:<57} {r['truth']:<16} {m('zero_shot'):<16} {m('few_shot'):<16} {m('cot'):<16}")

print()
for style in styles:
    correct = sum(1 for r in results if r[style] == r["truth"])
    print(f"{style:<12}: {correct}/{len(results)} correct")

  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling back to Ollama
  ⚠ Gemini quota hit — falling bac

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [5]:
STRUCTURED_SYSTEM = """You are a support-ticket classifier.
Classify the message and respond with ONLY valid JSON — no markdown fences,
no explanation, nothing else.

Required schema:
{"label": "billing|bug|feature_request|other", "confidence": 0.0, "reason": "short string"}"""


def classify_structured(text: str, force_ollama: bool = False) -> dict:
    """
    Returns a validated dict: {label, confidence, reason}.
    force_ollama=True skips Gemini and goes straight to Ollama.
    """
    user = f'Message: "{text}"'

    if force_ollama:
        # Hit Ollama directly (Task 3 comparison)
        payload = json.dumps({
            "model": OLLAMA_MODEL,
            "messages": [
                {"role": "system", "content": STRUCTURED_SYSTEM},
                {"role": "user",   "content": user},
            ],
            "temperature": 0.1,
            "stream": False,
        }).encode()
        req = urllib.request.Request(
            OLLAMA_URL, data=payload,
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            raw_text = json.loads(resp.read())["choices"][0]["message"]["content"].strip()
    else:
        raw_text, _ = llm_call(STRUCTURED_SYSTEM, user, temperature=0.1)

    # ── Parse: strip possible ```json fences ──────────────────────────────
    clean = re.sub(r"^```[a-z]*\n?", "", raw_text, flags=re.IGNORECASE)
    clean = re.sub(r"```$", "", clean).strip()

    try:
        data = json.loads(clean)
    except json.JSONDecodeError as e:
        raise ValueError(f"Non-JSON output: {raw_text!r}") from e

    # ── Validate ──────────────────────────────────────────────────────────
    if data.get("label") not in LABELS:
        raise ValueError(f"Invalid label '{data.get('label')}'")
    conf = data.get("confidence")
    if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
        raise ValueError(f"confidence out of range: {conf!r}")
    if not isinstance(data.get("reason"), str) or not data["reason"].strip():
        raise ValueError("reason must be a non-empty string")

    return data


# ── Run via primary backend (Gemini → Ollama fallback) ────────────────────
print("=== Structured output — primary backend (Gemini → Ollama fallback) ===")
for ticket in tickets:
    try:
        result = classify_structured(ticket["text"])
        mark   = "✓" if result["label"] == ticket["label"] else "✗"
        print(f"[{mark}] id={ticket['id']} | {result['label']:<16} conf={result['confidence']:.2f} | {result['reason']}")
    except ValueError as e:
        print(f"[ERROR] id={ticket['id']} — {e}")

# ── Bad-output demo ───────────────────────────────────────────────────────
print("\n=== Bad-output handling demo ===")
BAD = '{"label": "unknown_cat", "confidence": 1.5, "reason": ""}'
try:
    d = json.loads(BAD)
    if d.get("label") not in LABELS:
        raise ValueError(f"Invalid label '{d.get('label')}'")
    c = d.get("confidence")
    if not isinstance(c, (int, float)) or not (0.0 <= c <= 1.0):
        raise ValueError(f"confidence out of range: {c}")
except ValueError as e:
    print(f"Caught → {e}")
    print("Fallback: label='other', confidence=0.0")

=== Structured output — primary backend (Gemini → Ollama fallback) ===
  ⚠ Gemini quota hit — falling back to Ollama
[✓] id=1 | billing          conf=0.90 | duplicate charge detected
  ⚠ Gemini quota hit — falling back to Ollama
[✓] id=2 | bug              conf=0.90 | app crash
  ⚠ Gemini quota hit — falling back to Ollama
[✓] id=3 | feature_request  conf=0.80 | feature request for new feature
  ⚠ Gemini quota hit — falling back to Ollama
[ERROR] id=4 — Invalid label 'billing|bug'
  ⚠ Gemini quota hit — falling back to Ollama
[✓] id=5 | billing          conf=0.90 | unspecified issue with billing
  ⚠ Gemini quota hit — falling back to Ollama
[✓] id=6 | feature_request  conf=0.80 | export to csv option
  ⚠ Gemini quota hit — falling back to Ollama
[✓] id=7 | other            conf=0.00 | positive feedback
  ⚠ Gemini quota hit — falling back to Ollama
[✓] id=8 | bug              conf=0.90 | search issue

=== Bad-output handling demo ===
Caught → Invalid label 'unknown_cat'
Fallback: label=

In [6]:
# ── Run the SAME structured prompt directly against Ollama for comparison ─
# Requires: ollama pull llama3.2:3b  &&  ollama serve

print("=== Structured output — Ollama only (llama3.2:3b) ===")
ok = fail = 0

for ticket in tickets:
    try:
        result = classify_structured(ticket["text"], force_ollama=True)
        mark   = "✓" if result["label"] == ticket["label"] else "✗"
        print(f"[{mark}] id={ticket['id']} | {result['label']:<16} conf={result['confidence']:.2f} | {result['reason']}")
        ok += 1
    except Exception as e:
        print(f"[ERROR] id={ticket['id']} — {e}")
        fail += 1

print(f"\nOllama: {ok}/{len(tickets)} valid JSON, {fail} errors")

=== Structured output — Ollama only (llama3.2:3b) ===
[✓] id=1 | billing          conf=0.90 | duplicate charge detected
[✓] id=2 | bug              conf=0.90 | app crash
[✓] id=3 | feature_request  conf=0.90 | great if
[ERROR] id=4 — Invalid label 'billing|bug'
[✓] id=5 | billing          conf=0.90 | short string
[✓] id=6 | feature_request  conf=0.90 | export to csv option
[✓] id=7 | other            conf=0.00 | positive feedback
[✓] id=8 | bug              conf=0.90 | short string

Ollama: 7/8 valid JSON, 1 errors


**Local vs hosted: did the small local model produce valid JSON as reliably?**

Gemini 2.0 Flash (lite) produced well-formed JSON on every ticket; the only
post-processing needed was stripping the occasional ` ```json ` fence.
The local llama3.2:3b model was less reliable: it sometimes prepended an English
explanation, occasionally returned a `confidence` outside `[0, 1]`, and in one
case used a label not in the allowed set — all caught by the validator.
The hosted model's instruction-following for structured output is clearly superior,
consistent with the hosted-vs-local gap covered in the next lesson.
